# Denoiser Full-Track Waterfall

Reads the full I/Q track for a test-set event, applies the pretrained
`LitS4DenoisingModel` in non-overlapping windows of `cutoff` samples,
and produces a time vs. frequency waterfall plot coloured by
scaled signal-to-noise ratio.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import h5py
import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq, fftshift

from src.models.model import LitS4DenoisingModel
from src.models.networks import S4DSeq2SeqModel
from src.utils.noise import noise_model, bandpass_filter

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Configuration

Set the paths below before running the notebook.

In [ ]:
# ── Update these paths ────────────────────────────────────────────────────────
CHECKPOINT_PATH = '../runs/denoising/lightning_logs/<run_id>/checkpoints/last.ckpt'
CONFIG_PATH     = '../runs/denoising/config.yaml'
DATA_DIR        = '/path/to/data/'   # directory containing HDF5 files

# Index of the test-set track to visualise (0 = first test track)
TEST_TRACK_IDX  = 0

# Maximum number of segments to process (None = all that fit in the track)
MAX_SEGMENTS    = None

# Baseband half-width to display around the carrier frequency [Hz]
DISPLAY_BW_HZ   = 6e6   # ±6 MHz → 12 MHz total display bandwidth

# Sampling rate (must match noise_model in src/utils/noise.py)
FS = 403e6   # Hz
# ─────────────────────────────────────────────────────────────────────────────

## 2. Load Config and Model

In [ ]:
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

data_cfg  = config['data']['init_args']
model_cfg = config['model']['init_args']
enc_cfg   = model_cfg['encoder']['init_args']

INPUTS       = data_cfg['inputs']                    # ['output_ts_I', 'output_ts_Q']
CUTOFF       = data_cfg.get('cutoff', 8192)          # samples per segment
NOISE_CONST  = data_cfg.get('noise_const', 1.0)
APPLY_FILTER = data_cfg.get('apply_filter', False)

print(f'Inputs        : {INPUTS}')
print(f'Cutoff        : {CUTOFF} samples  ({CUTOFF / FS * 1e6:.1f} µs per segment)')
print(f'Noise const   : {NOISE_CONST}')
print(f'Apply filter  : {APPLY_FILTER}')

In [ ]:
encoder = S4DSeq2SeqModel(
    d_input  = enc_cfg['d_input'],
    d_output = enc_cfg['d_output'],
    d_model  = enc_cfg.get('d_model', 128),
    n_layers = enc_cfg.get('n_layers', 6),
    dropout  = enc_cfg.get('dropout', 0.0),
    prenorm  = enc_cfg.get('prenorm', False),
)

model = LitS4DenoisingModel.load_from_checkpoint(
    CHECKPOINT_PATH,
    encoder=encoder,
    map_location=device,
).to(device).eval()

print('Model loaded successfully')
print(f'  d_model={enc_cfg.get("d_model", 128)}, n_layers={enc_cfg.get("n_layers", 6)}')

## 3. Identify the Test-Set Track

Reproduce the exact same 80/10/10 split (seed 42) used during training so that
we evaluate on a track the model has never seen.

In [ ]:
import os

hdf5_files = sorted([
    os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.endswith('.hdf5') or f.endswith('.h5')
])

# Build global index (file_path, local_row_idx) for every track
global_index = []
for path in hdf5_files:
    with h5py.File(path, 'r') as f:
        n = f[INPUTS[0]].shape[0]
    global_index.extend((path, i) for i in range(n))

total_tracks = len(global_index)
print(f'Total tracks in dataset : {total_tracks}')

# Replicate the training split (same seed / ratios as LitDenoisingDataModule)
rng_split = torch.Generator().manual_seed(42)
train_ds, val_ds, test_ds = torch.utils.data.random_split(
    range(total_tracks), [0.8, 0.1, 0.1], generator=rng_split
)
test_indices = list(test_ds.indices)
print(f'Test-set size           : {len(test_indices)} tracks')

# Resolve the requested test track
global_row = test_indices[TEST_TRACK_IDX]
hdf5_path, local_row = global_index[global_row]
print(f'\nTest track {TEST_TRACK_IDX}  →  file: {os.path.basename(hdf5_path)},  row: {local_row}')

## 4. Read the Full Track

In [ ]:
with h5py.File(hdf5_path, 'r') as f:
    track_I = f[INPUTS[0]][local_row][:].astype(np.float32)   # full I channel
    track_Q = f[INPUTS[1]][local_row][:].astype(np.float32)   # full Q channel

    # Carrier frequency for physical frequency axis (optional metadata field)
    if 'avg_carrier_frequency_Hz' in f:
        carrier_freq_hz = float(f['avg_carrier_frequency_Hz'][local_row])
    else:
        carrier_freq_hz = 0.0   # fall back to baseband-only axis

track_len   = len(track_I)
n_segments  = track_len // CUTOFF
if MAX_SEGMENTS is not None:
    n_segments = min(n_segments, MAX_SEGMENTS)

track_duration_s = track_len / FS

print(f'Track length    : {track_len:,} samples  ({track_duration_s*1e3:.3f} ms)')
print(f'Segments        : {n_segments}  ×  {CUTOFF} samples')
print(f'Carrier freq    : {carrier_freq_hz/1e9:.6f} GHz')

## 5. Apply Denoiser Segment by Segment

In [ ]:
denoised_I = np.zeros(n_segments * CUTOFF, dtype=np.float32)
denoised_Q = np.zeros(n_segments * CUTOFF, dtype=np.float32)

for s in range(n_segments):
    start = s * CUTOFF
    end   = start + CUTOFF

    seg_I = track_I[start:end].copy()
    seg_Q = track_Q[start:end].copy()

    # --- add noise and normalise (mirrors Project8SimDenoising.__getitem__) ---
    X_noisy = np.stack([seg_I, seg_Q], axis=-1)   # (cutoff, 2)
    X_norm  = np.zeros_like(X_noisy)
    norms   = np.zeros(2, dtype=np.float32)

    for j in range(2):
        noise = noise_model(CUTOFF, NOISE_CONST)
        Xn    = X_noisy[:, j] + noise
        if APPLY_FILTER:
            Xn = bandpass_filter(Xn)
        s_std       = float(np.std(Xn)) + 1e-8
        norms[j]    = s_std
        X_norm[:, j] = Xn / s_std

    # --- run denoiser --------------------------------------------------------
    x_tensor = torch.tensor(X_norm, dtype=torch.float32).unsqueeze(0).to(device)  # (1, L, 2)
    with torch.no_grad():
        x_pred_norm = model(x_tensor).squeeze(0).cpu().numpy()   # (L, 2)

    # --- denormalise to original amplitude scale ----------------------------
    denoised_I[start:end] = x_pred_norm[:, 0] * norms[0]
    denoised_Q[start:end] = x_pred_norm[:, 1] * norms[1]

    if (s + 1) % max(1, n_segments // 10) == 0:
        print(f'  processed {s+1:4d} / {n_segments} segments')

print('Done.')

## 6. Compute Power Spectra

Treat each segment as a complex sample `I + jQ`, compute the FFT power
spectrum, and estimate the per-segment noise floor from the median power.

In [ ]:
# Baseband frequency axis (shifted so DC is at centre)
baseband_freqs_hz = fftshift(fftfreq(CUTOFF, d=1.0 / FS))   # Hz
physical_freqs_ghz = (carrier_freq_hz + baseband_freqs_hz) / 1e9  # GHz

# Select a narrow band around the carrier for display
freq_mask = np.abs(baseband_freqs_hz) <= DISPLAY_BW_HZ

# Time centres of each segment (seconds)
seg_dt_s     = CUTOFF / FS
seg_times_s  = np.arange(n_segments) * seg_dt_s + seg_dt_s / 2

# Build 2-D power array: shape (n_display_freqs, n_segments)
n_display = int(freq_mask.sum())
power_map  = np.zeros((n_display, n_segments), dtype=np.float32)

for s in range(n_segments):
    start    = s * CUTOFF
    end      = start + CUTOFF
    cplx     = denoised_I[start:end] + 1j * denoised_Q[start:end]
    spectrum = fftshift(np.abs(fft(cplx)) ** 2)
    power_map[:, s] = spectrum[freq_mask]

# Scale each segment by its noise floor (median power across displayed bins)
noise_floor       = np.median(power_map, axis=0, keepdims=True)  # (1, n_segments)
noise_floor       = np.where(noise_floor > 0, noise_floor, 1.0)
snr_map           = power_map / noise_floor                       # ≈1 for pure noise

# Clip and normalise to [0, 1] (values >> 1 only where signal is present)
snr_max  = np.percentile(snr_map, 99.9)
snr_map_scaled = np.clip(snr_map / snr_max, 0.0, 1.0)

print(f'Power array shape  : {power_map.shape}')
print(f'Freq range displayed : {physical_freqs_ghz[freq_mask].min():.4f} – '
      f'{physical_freqs_ghz[freq_mask].max():.4f} GHz')
print(f'SNR 99.9th pct (= vmax) : {snr_max:.2f}')

## 7. Waterfall Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# pcolormesh expects edge arrays one element longer than the data
t_edges = np.append(seg_times_s - seg_dt_s / 2, seg_times_s[-1] + seg_dt_s / 2)
f_ghz   = physical_freqs_ghz[freq_mask]
df_ghz  = np.abs(f_ghz[1] - f_ghz[0])
f_edges = np.append(f_ghz - df_ghz / 2, f_ghz[-1] + df_ghz / 2)

mesh = ax.pcolormesh(
    t_edges,
    f_edges,
    snr_map_scaled,          # shape (n_display_freqs, n_segments)
    cmap='Blues',
    vmin=0.0,
    vmax=1.0,
    shading='flat',
    rasterized=True,
)

cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
cbar.set_label('Scaled signal to noise ratio (linear)', fontsize=11)
cbar.set_ticks(np.linspace(0.1, 1.0, 10))

ax.set_xlabel('Time [s]', fontsize=12)
ax.set_ylabel('Frequency [GHz]', fontsize=12)
ax.set_title(f'Project 8 – Event {TEST_TRACK_IDX}', fontsize=13)

fig.tight_layout()
plt.show()

## 8. Side-by-side Comparison: Noisy vs. Denoised

Repeat the same plot using the *noisy* (pre-denoising) signal for comparison.

In [ ]:
# Build noisy power map using the raw track (with fresh noise added)
noisy_power_map = np.zeros((n_display, n_segments), dtype=np.float32)

for s in range(n_segments):
    start = s * CUTOFF
    end   = start + CUTOFF
    I_raw = track_I[start:end] + noise_model(CUTOFF, NOISE_CONST)
    Q_raw = track_Q[start:end] + noise_model(CUTOFF, NOISE_CONST)
    cplx  = I_raw + 1j * Q_raw
    spec  = fftshift(np.abs(fft(cplx)) ** 2)
    noisy_power_map[:, s] = spec[freq_mask]

noisy_noise_floor  = np.median(noisy_power_map, axis=0, keepdims=True)
noisy_noise_floor  = np.where(noisy_noise_floor > 0, noisy_noise_floor, 1.0)
noisy_snr          = noisy_power_map / noisy_noise_floor
noisy_snr_max      = np.percentile(noisy_snr, 99.9)
noisy_snr_scaled   = np.clip(noisy_snr / noisy_snr_max, 0.0, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

for ax, data, title in [
    (axes[0], noisy_snr_scaled,  'Noisy input'),
    (axes[1], snr_map_scaled,    'Denoised'),
]:
    mesh = ax.pcolormesh(
        t_edges, f_edges, data,
        cmap='Blues', vmin=0.0, vmax=1.0,
        shading='flat', rasterized=True,
    )
    cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
    cbar.set_label('Scaled signal to noise ratio (linear)', fontsize=10)
    cbar.set_ticks(np.linspace(0.1, 1.0, 10))
    ax.set_xlabel('Time [s]', fontsize=12)
    ax.set_ylabel('Frequency [GHz]', fontsize=12)
    ax.set_title(f'Project 8 – Event {TEST_TRACK_IDX} ({title})', fontsize=12)

fig.tight_layout()
plt.show()